# How small can a world be and still keep its air? Find a planet that could test it

[Open this notebook in Colab](https://colab.research.google.com/github/Rakiura-Inc/limits-of-habitability/blob/main/notebooks/01_find_your_test_planet.ipynb)

The lab's model says a rocky planet at Earth's distance from a sun-like star has to be at least 0.8 Earth radii to still have an atmosphere after a billion years. That is a prediction. Part 1 looks for the real planets that come closest to testing it, and Part 2 works out whether the one you pick can be weighed.

Run the cells in order (Shift+Enter). Boxes headed **Write down** ask you questions. Answer each one in the cell underneath by replacing its `...`, then run that cell: the notebook keeps your answer. At the end of each part, one cell puts all your answers into a single table and saves it with your plots. That table is what you hand in on the step. Part 2 uses the planet you pick in Part 1.

In [ ]:
# Setup. Run this first. In Colab it installs what the notebook needs.
# If Colab asks you to restart the session afterwards, restart it and run the notebook from the top.
import importlib.util

# habtrain: a small package written for these notebooks. It downloads the data, fits orbits and draws the plots.
if importlib.util.find_spec("habtrain") is None:
    %pip install -q "habtrain @ git+https://github.com/Rakiura-Inc/limits-of-habitability@main"

# rakiura: the optional helper that connects this notebook to your step in Rakiura.
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"

import numpy as np                 # arrays and arithmetic
import matplotlib.pyplot as plt    # plots
import habtrain as ht              # the notebooks' package, used as ht.data, ht.fit, ht.plot and so on

# Which notebook this is, so your hand-in records exactly what you ran.
NOTEBOOK = "notebooks/01_find_your_test_planet.ipynb"
MODE = ht.portal.connect(NOTEBOOK)

> **Connect to Rakiura (optional)**
>
> Connect, and Rakiura ticks off each part as you finish it, with your plots on the step. You still attach your answers table and press Hand in on the step yourself. Skip it and everything still works: you attach the saved files on the step.
>
> To connect, run the cell below, open the link it prints, check the code matches the one shown, and click Allow. In Colab you do this once per session. To skip the click next time, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; your key is in Rakiura under Settings > Notebook key).

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = ht.portal.connect(NOTEBOOK, sign_in=True)

## Part 1: Find a planet that could test the lab's size limit

### What the lab found

In 1993 Kasting, Whitmire and Reynolds worked out the habitable zone: the band of orbits around a star where a planet like Earth could keep liquid water on its surface ([Kopparapu et al. 2013](https://arxiv.org/abs/1301.6674) explains how its edges are set). It says where a planet has to orbit. It takes for granted what the planet is: a world still warm enough inside to keep cycling carbon dioxide between its rocks and its air, which is what holds a climate steady over billions of years.

A small planet cools faster inside. At some size it can no longer put gas back into its air as fast as the air escapes to space, and the atmosphere is lost. The lab built a model that follows both over time: the Smaller Than Earth Habitability Model, STEHM ([Hill, Kane, Foley and Schaefer 2026](https://doi.org/10.3847/PSJ/ae6804)).

The model's planet is rocky with a **stagnant lid**, one unbroken outer shell and no plate tectonics, the way every rocky planet in the solar system except Earth is thought to work. It sits at Earth's distance from a sun-like star, and its air is pure carbon dioxide, the best case for keeping an atmosphere. Under Earth-like starting conditions it has to be at least **0.8 Earth radii** to still have an atmosphere after a billion years. Change the starting conditions and the line moves. The carbon a planet starts with moves it most. The model's default carbon sits inside the range estimated for Earth. With about six times the default, planets down to 0.6 Earth radii hold on, and at the lowest estimate of Earth's carbon the line rises to 0.9. At the outer edge of the habitable zone it drops to 0.7.

That number is a prediction about real planets. Part 1 finds the ones that could test it.

### The catalog

The cell below loads the lab's catalog of every known planet that spends time in its star's habitable zone ([Hill et al. 2023](https://arxiv.org/abs/2210.02484)). Each row is one planet. The columns you will use:

- `Rad`: the planet's radius, in Earth radii
- `f_Rad`: 1 if the radius was estimated rather than measured (Question 2 explains)
- `Mass` and `f_Mass`: the same for its mass, in Earth masses; `l_Mass` is `<=` or `>=` where the mass is only a limit
- `S`: the starlight reaching it, in watts per square metre (Earth gets 1,361; the plots divide by 1,361, so Earth is 1)
- `Per`: its orbital period, in days
- `e`: how stretched its orbit is, from 0 for a circle towards 1 for a long thin oval
- `Mstar` and `Rstar`: its star's mass and radius, in Sun units
- `CHZ` and `OHZ`: the percentage of its orbit spent in the conservative and in the optimistic habitable zone
- `Teff`: its star's surface temperature, in kelvin (the Sun's is 5,772)
- `Vmag`: how bright its star looks from here; a bigger number is fainter
- `K`: the wobble the planet should cause in its star, in metres per second (Part 2)

In [ ]:
# Load the lab's habitable zone catalog: one row per planet (it downloads once, then reads from data/).
cat = ht.data.load_catalog()
print(len(cat), "planets")

# Show the first five rows, to see what the columns look like.
cat.head()

### Every planet on one plot

Radius against the starlight each planet gets, with the lab's 0.8 Earth radii line. Before you go on, look at three things the next questions use:
- the flat band of grey points along the top: radii the catalog estimated from the mass (Question 2)
- the red rings: planets with both the mass and the radius measured (Question 2)
- the planets near the dashed 0.8 line (Question 4)

In [ ]:
# Every planet's radius against the starlight it gets, with the lab's 0.8 Earth radii line.
fig = ht.plot.catalog(cat)
plt.show()

### Question 1: conservative or optimistic?

**What you need**

The habitable zone has two common versions, both from Kopparapu et al. 2013 and 2014.
- The **conservative zone** runs from the runaway greenhouse limit, inside which a planet's oceans would boil away, out to the maximum greenhouse limit, beyond which even a thick carbon dioxide atmosphere cannot keep it warm.
- The **optimistic zone** is wider. It runs from the recent Venus limit out to the early Mars limit, set by evidence that both planets once had liquid water on their surfaces.

`CHZ` and `OHZ` hold the percentage of each planet's orbit spent inside each zone. A planet that never leaves has 100.

> **Write down: how many planets stay in each zone for their whole orbit**
>
> - How many planets spend their whole orbit in the conservative zone? Set `whole_orbit_conservative` to the number of planets whose `CHZ` is 100.
> - How many spend their whole orbit in the optimistic zone? Set `whole_orbit_optimistic` to the number whose `OHZ` is 100.
>
> **Where to look:** the `CHZ` and `OHZ` columns. A planet that never leaves a zone has 100.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

`cat[cat.OHZ == 100]` keeps only the planets with 100 in the `OHZ` column.

</details>

<details><summary>Stuck? Open hint 2</summary>

`len(...)` counts what is left.

</details>

In [ ]:
# Your code: count the planets that spend 100% of their orbit inside each zone.
whole_orbit_conservative = ...   # how many planets have CHZ == 100
whole_orbit_optimistic = ...     # how many planets have OHZ == 100

# Keep both answers for your hand-in table.
ht.record.save("zone_counts", {"conservative": whole_orbit_conservative, "optimistic": whole_orbit_optimistic},
               "planets in each zone for their whole orbit")

# A sense check: the conservative zone is the narrower of the two, so it cannot hold more planets.
if ht.record.filled(whole_orbit_conservative) and ht.record.filled(whole_orbit_optimistic) \
        and whole_orbit_conservative > whole_orbit_optimistic:
    print("The conservative zone is the narrower one, so it should hold fewer planets. Check which column you used.")

### Question 2: measured or estimated?

**What you need**

A planet seen crossing in front of its star (a transit) has a measured radius. A planet found by the wobble it causes in its star has a mass from data, but usually a minimum mass: the wobble only shows the part of the star's motion along our line of sight. If the planet also transits, that is its true mass. A few masses in the catalog are only limits, marked in `l_Mass`: count those as not measured. So "measured" here means `f_Mass` is 0 and `l_Mass` is empty. Most of these are minimum masses, which the catalog still counts as measured. Where one of the two is missing, the catalog estimates it from a mass-radius relation, a fit to the planets that have both ([Chen and Kipping 2017](https://scixplorer.org/abs/2017ApJ...834...17C/abstract)). `f_Rad` and `f_Mass` are 1 where the number is an estimate.

A planet's density needs both numbers measured. The relation also predicts almost the same radius for every planet heavier than about 0.4 Jupiter masses, because giant planets of very different masses are all close to Jupiter's size. That is why the grey points along the top of the first plot sit in a flat band: their radii are estimates from the mass, not measurements.

> **Write down: how many planets have each number measured**
>
> - How many planets have a measured radius? Put it in `radius_measured`.
> - How many have a measured mass? Put it in `mass_measured`.
> - How many have both? Put it in `both_measured`.
>
> **Where to look:** the `f_Rad` and `f_Mass` columns, and the grey band on the first plot.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

`(cat.f_Rad == 0).sum()` counts the planets whose radius is measured. For mass, leave out the limits too: `(cat.f_Mass == 0) & (cat.l_Mass == "")`.

</details>

<details><summary>Stuck? Open hint 2</summary>

Combine conditions with `&`, each in brackets: `(cat.f_Rad == 0) & (cat.f_Mass == 0) & (cat.l_Mass == "")`.

</details>

In [ ]:
# Your code: count planets by which numbers are measured (f_Rad and f_Mass are 0 when measured).
radius_measured = ...   # planets whose radius is measured
mass_measured = ...     # planets whose mass is from data and not just a limit
both_measured = ...     # planets with both measured

# Keep the answers for your hand-in table.
ht.record.save("measured_counts", {"radius": radius_measured, "mass": mass_measured, "both": both_measured},
               "planets with a measured radius, a measured mass, and both")

# A sense check: "both measured" can never be more than either count on its own.
if all(ht.record.filled(x) for x in (radius_measured, mass_measured, both_measured)) \
        and both_measured > min(radius_measured, mass_measured):
    print("Both measured cannot be more than either one alone. Check the & in your filter.")

### Question 3: what kind of star?

**What you need**

The Sun's surface is about 5,772 K. Stars cooler than about 4,000 K are mostly M dwarfs: small, cool and red. The lab's 0.8 Earth radii line was worked out for a planet around a sun-like star.

The plot below colours each planet by its star's temperature.

In [ ]:
# The same plot, with each planet coloured by its star's surface temperature.
fig = ht.plot.catalog(cat, color_by="Teff", title="The same planets, coloured by their star's temperature")
plt.show()

> **Write down: what kind of star the small habitable zone planets orbit**
>
> - Make a table of the planets under 2 Earth radii that stay in the optimistic zone their whole orbit. Put the code in `small`.
> - How many planets are in it? Put it in `n_small`.
> - How many of them orbit a star cooler than 4,000 K? Put it in `n_small_cool`.
> - In one line: what kind of star do most of them orbit? Put it in `STAR_KIND`.
>
> **Where to look:** the coloured plot above, and the `Rad`, `OHZ` and `Teff` columns.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

`cat[(cat.Rad < 2) & (cat.OHZ == 100)]` keeps the small planets that never leave the optimistic zone.

</details>

<details><summary>Stuck? Open hint 2</summary>

`(small.Teff < 4000).sum()` counts the ones around cool stars.

</details>

In [ ]:
# Your code: the small planets that stay in the optimistic zone for their whole orbit.
small = ...          # a table: planets with Rad under 2 and OHZ equal to 100
n_small = ...        # how many there are
n_small_cool = ...   # how many orbit a star cooler than 4,000 K
STAR_KIND = ...      # one line, in quotes

# Keep the answers for your hand-in table.
ht.record.save("small_counts", {"small planets": n_small, "around stars under 4,000 K": n_small_cool},
               "small planets in the optimistic zone for their whole orbit, and how many orbit cool stars")
ht.record.save("star_kind", STAR_KIND, "what kind of star most of them orbit")

# Later cells need this table. If you have not filled it in yet, a stand-in is used so they still run.
small_planets = ht.record.fill(small, ht.data.small_hz_planets(cat), "small")

### Question 4: who sits near the line?

**Predict first**

How many of the 328 planets do you expect between 0.6 and 1.0 Earth radii, either side of the lab's line? Look at the first plot again if you like, then write a number.

In [ ]:
# Your prediction, before you look: how many planets between 0.6 and 1.0 Earth radii?
MY_GUESS = ...   # a whole number, or "unsure"

# Saved with the time, so your hand-in shows you predicted first.
ht.record.predict("near_line", MY_GUESS)

Now find them. **What you need:** look back at "What the lab found" at the top of this notebook for the kind of planet and star the lab's model assumes. The paper sets out the model in full ([Hill, Kane, Foley and Schaefer 2026](https://doi.org/10.3847/PSJ/ae6804)).

> **Write down: the planets near the line, and whether the lab's model applies to them**
>
> - Make a table of the planets with a radius from 0.6 to 1.0 Earth radii. Put the code in `near`.
> - Which planets are they? Their names, in quotes. Put them in `NEAR_LINE`.
> - Does the lab's 0.8 line apply to them? Yes or no, and why, in one line. Put it in `MODEL_APPLIES`.
>
> **Where to look:** the table your code shows, especially each star's `Teff`.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

`cat[(cat.Rad >= 0.6) & (cat.Rad <= 1.0)]`

</details>

In [ ]:
# This only runs once you have made your prediction above.
if ht.record.revealed("near_line"):
    near = ...   # your code: the planets with Rad from 0.6 to 1.0
    # Show them with the columns that matter here.
    ht.record.show(near, ["Planet", "Rad", "f_Rad", "Mass", "f_Mass", "OHZ", "Teff"])

NEAR_LINE = ...       # their names, in quotes
MODEL_APPLIES = ...   # one line, in quotes: does the 0.8 line apply to them, and why?

# Keep the answers for your hand-in table.
ht.record.save("near_line", NEAR_LINE, "planets between 0.6 and 1.0 Earth radii")
ht.record.save("model_applies", MODEL_APPLIES, "whether the lab's line applies to them, and why")

### Question 5: pick your planet

There is no right answer, and no planet yet tests the line cleanly: that is part of what you have found. Pick the one you think comes closest, or the one you most want to spend time on, from the small planets in Question 3: they include the ones next to the line. The map below numbers each one. Choose from the menu under it, and the map circles your choice. If you would rather not choose, the first option picks one at random.

In [ ]:
# The small habitable zone planets on the radius-against-starlight plot, each numbered.
candidates = small_planets.reset_index(drop=True)
fig = ht.plot.catalog(candidates, numbers=True, title="Small planets that stay in the optimistic zone for their whole orbit")
plt.show()

# A menu of the same planets. Choosing one redraws the plot with your choice circled.
describe = lambda n, r: (f"{n}. {r.Planet}: {r.Rad} Earth radii ({'estimated' if r.f_Rad else 'measured'}), "
                         f"mass {'estimated' if r.f_Mass else 'measured'}, star {r.Teff:.0f} K")
draw = lambda t, mark: ht.plot.catalog(t, numbers=True, highlight=[mark], title="Your choice, circled")
chooser = ht.choose.menu(candidates, "Planet", describe, draw, "Your planet:")

> **Write down: your planet**
>
> - Choose your planet in the menu above. If the menu does not appear, put its number from the plot in `MY_PICK`.
> - Why this planet, and could it test the lab's size limit? One or two lines. Put it in `WHY`.
> - What is still missing before anyone could test the limit with it: a measurement, or something about its star? Name it: not "more data". Put it in `MISSING`.
>
> **Where to look:** the plot and the menu above.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
MY_PICK = ...   # only if the menu did not appear: your planet's number on the plot
WHY = ...       # one or two lines, in quotes
MISSING = ...   # the missing measurement, in quotes

# Your planet: what you chose in the menu, else the number you typed, else a random one.
MY_PLANET = ht.choose.resolve(chooser, MY_PICK, candidates, "Planet", "planet")

# Keep the answers for your hand-in table, and show your planet's row of the catalog.
ht.record.save("my_planet", MY_PLANET, "your planet")
ht.record.save("why", WHY, "why this planet, and whether it could test the limit")
ht.record.save("missing", MISSING, "the measurement still missing")
ht.record.show(cat[cat.Planet == MY_PLANET])

### Hand in Part 1

Run the cell below. It shows all your answers from Part 1 in one table. Check them, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `planet-picked`, and download them.

In [ ]:
# Your planet marked on the full plot, saved with your answers as your Part 1 hand-in.
fig = ht.plot.catalog(cat, highlight=[MY_PLANET])
ht.record.handin("planet-picked",
                 ["zone_counts", "measured_counts", "small_counts", "star_kind",
                  "near_line", "model_applies", "my_planet", "why", "missing"],
                 figures={"catalog": fig})

## Part 2: Can your planet be weighed?

### How a planet is weighed

A planet and its star both orbit the point between them. The star swings a little, and that swing is all you can see: its light shifts slightly blue as it comes towards you and slightly red as it moves away. That speed along your line of sight is the **radial velocity**. **K**, the semi-amplitude, is how far the star's speed swings either side of its average; it carries the planet's mass. For a circular orbit:

K = 28.4 m/s × (planet mass in Jupiter masses) × (period in years)<sup>-1/3</sup> × (star mass in solar masses)<sup>-2/3</sup>

An Earth mass is about 1/318 of Jupiter's, so in Earth masses the 28.4 m/s becomes about 0.0895 m/s. For a stretched, oval orbit (column `e` above 0), divide by the square root of (1 - e<sup>2</sup>).

Earth moves the Sun by about 9 centimetres per second, and Jupiter moves it by about 12.5 metres per second. A **spectrograph** is the instrument that measures a star's velocity from its light. ESPRESSO on the Very Large Telescope is one of the most precise working: its velocities of very bright, quiet stars scatter by about 40 centimetres per second over three and a half years ([Figueira et al. 2025](https://arxiv.org/abs/2507.07514)). This notebook calls that the **floor**: a wobble smaller than 40 cm/s (0.4 in the catalog's `K` column, which is in m/s) is out of reach even on the best stars. On faint stars the floor is higher, as Question 9 shows.

In [ ]:
# Your planet from Part 1. In a new session, run Part 1's cells first and put your planet's number in MY_PICK
# in Question 5 (the number was printed when you chose), so Part 2 weighs the same planet you handed in.
if "MY_PLANET" not in dir():
    raise SystemExit("Run Part 1's cells first, with your planet's number in MY_PICK in Question 5, then run this cell again.")
planet = MY_PLANET
row = cat[cat.Planet == planet].iloc[0]

# Print what Part 2 works from, and whether each number is measured or estimated.
print(f"{planet}: mass {row.Mass} Earth masses ({'estimated' if row.f_Mass else 'measured'}), "
      f"radius {row.Rad} Earth radii ({'estimated' if row.f_Rad else 'measured'}), "
      f"period {row.Per} days, star {row.Mstar} solar masses, e = {row.e}")

### Question 6: how big is its wobble?

Write the formula as a function, check it on Earth and Jupiter, then use it on your planet and compare with the catalog's own `K`.

> **Write down: your planet's wobble**
>
> - Fill in `my_wobble` so it returns K in m/s for any planet. The cell then checks it on Earth (it should give about 0.09 m/s) and on Jupiter (about 12.5 m/s), works out your planet's wobble, and prints the catalog's own `K` next to yours.
>
> **Where to look:** the formula above. For the checks: Earth is 1 Earth mass, 365.25 days, 1 solar mass; Jupiter is 317.8 Earth masses, 4,332.6 days, 1 solar mass.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

Work in Earth masses: `0.0895 * mass_earth * (period_days / 365.25) ** (-1/3) * star_mass_sun ** (-2/3)`.

</details>

In [ ]:
# Your code: the formula above as a function. It returns K in metres per second.
def my_wobble(mass_earth, period_days, star_mass_sun):
    return ...   # K in m/s, from the formula above

# Use it on your planet, then check it on Earth and Jupiter, whose answers you know.
mine = my_wobble(row.Mass, row.Per, row.Mstar)
if ht.record.filled(mine):
    print(f"Earth check: {my_wobble(1, 365.25, 1):.3f} m/s   Jupiter check: {my_wobble(317.8, 4332.6, 1):.2f} m/s")
    print(f"{planet}: yours {mine:.2f} m/s, the catalog's {row.K} m/s")
    if row.e and row.e > 0.1:
        print("This orbit is eccentric, so a small difference from the catalog's number is expected.")

# Keep both numbers for your hand-in table.
ht.record.save("wobble", {"yours (m/s)": mine, "catalog (m/s)": row.K}, "your planet's wobble, yours and the catalog's")

### Question 7: could anyone see it?

**Predict first**

Earth around the Sun sits under ESPRESSO's floor of about 40 centimetres per second. Think back to the kind of star you found in Question 3. A cool star's habitable zone sits much closer in, so planets there have short orbits, and the formula shows that a shorter period and a lighter star both make a bigger wobble. Before you plot the small planets: will most of their wobbles sit above 40 centimetres per second, or below?

In [ ]:
# Your prediction, before the plot: will most small planets sit above or below 40 cm/s?
MY_PREDICTION = ...   # "above", "below" or "unsure"
ht.record.predict("floor", MY_PREDICTION)

In [ ]:
# This only runs once you have made your prediction above.
if ht.record.revealed("floor"):
    # One bar per small planet, sorted by its predicted wobble; the dashed line is 40 cm/s.
    fig_floor = ht.plot.wobbles(small_planets, floor=0.4, mine=planet)
    plt.show()

> **Write down: how the small planets compare with the floor**
>
> - How many of the small planets have a wobble of 0.4 m/s or more? Put the code in `n_above`.
> - Is your planet above or below the floor, and by roughly how many times? One line. Put it in `MINE_VS_FLOOR`.
>
> **Where to look:** the bar chart: each bar is one planet, and the dashed line is 40 cm/s.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

`(small_planets.K >= 0.4).sum()`

</details>

In [ ]:
n_above = ...         # your code: how many small planets have K of 0.4 m/s or more
MINE_VS_FLOOR = ...   # one line, in quotes

# Keep the answers for your hand-in table.
ht.record.save("above_floor", n_above, "small planets with a wobble of 40 cm/s or more")
ht.record.save("mine_vs_floor", MINE_VS_FLOOR, "your planet against the floor")

### Question 8: what is it made of?

**What you need**

Density is mass divided by volume. In Earth units:

density ÷ 5.514 g/cm<sup>3</sup> = (mass in Earth masses) ÷ (radius in Earth radii)<sup>3</sup>

Earth's density is 5.514 g/cm<sup>3</sup> and Neptune's 1.638 g/cm<sup>3</sup> ([NASA planetary fact sheet](https://nssdc.gsfc.nasa.gov/planetary/factsheet/)). Neptune is under a third of Earth's because it is mostly light material: a deep layer of water, methane and ammonia under a hydrogen and helium atmosphere, around a small rocky core. A density only means something if both the mass and the radius are measured.

> **Write down: your planet's density, or why it cannot have one**
>
> - If your planet's mass and radius are both measured: what is its density in g/cm<sup>3</sup>? Put it in `MY_DENSITY`. If either is estimated, leave `MY_DENSITY` as `...`.
> - What is your planet probably made of? Or, if it has no density, which number is estimated and what a density would rest on. Put it in `MADE_OF`.
>
> **Where to look:** your planet's `Mass`, `Rad`, `f_Mass` and `f_Rad`, printed at the start of Part 2.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
MY_DENSITY = ...   # g/cm3, only if both mass and radius are measured; otherwise leave ...
MADE_OF = ...      # one line, in quotes

# Keep the answers for your hand-in table.
ht.record.save("density", MY_DENSITY, "your planet's density (g/cm3)")
ht.record.save("made_of", MADE_OF, "what it is made of, or why there is no density")

### Question 9: what is in the way?

The 40 cm/s floor is for the brightest, quietest stars. Count how many of the small planets have a measured mass, then compare their wobbles with the ones that do not, and look at your own planet's star.

**What you need**

A fainter star sends fewer photons, so each velocity is less precise and takes longer to get. A star of magnitude 15 is about 250 times fainter than one of magnitude 9.

A few planets are weighed another way. In a system with several transiting planets, their pulls on each other shift when each transit happens by minutes, and those shifts give their masses without any wobble.

> **Write down: what stops a planet being weighed**
>
> - How many of the small planets have a measured mass? Put the code in `n_weighed`.
> - What stops the rest being weighed? Compare their wobbles with the weighed ones, and their stars' brightness. One or two lines. Put it in `IN_THE_WAY`.
>
> **Where to look:** `f_Mass` for the small planets, and `Vmag` and `Teff` for your planet's star.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

`((small_planets.f_Mass == 0) & (small_planets.l_Mass == "")).sum()`, then compare `small_planets.K` for the two groups.

</details>

<details><summary>Stuck? Open hint 2</summary>

Compare your star's `Vmag` with the rest of the catalog: `cat.Vmag.describe()`.

</details>

In [ ]:
# Your planet's star first: how bright and how hot it is.
print(f"Your planet's star: Vmag {row.Vmag}, Teff {row.Teff} K")

n_weighed = ...    # your code: how many small planets have a mass from data (f_Mass of 0, and not a limit)
IN_THE_WAY = ...   # one or two lines, in quotes

# Keep the answers for your hand-in table, then look at your own planet's star.
ht.record.save("weighed", n_weighed, "small planets with a measured mass")
ht.record.save("in_the_way", IN_THE_WAY, "what stops a planet being weighed")

### Hand in Part 2

Run the cell below. It shows all your answers from Part 2 in one table. Check them, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `planet-weighed`, and download them.

In [ ]:
# Your Part 2 answers, and the wobble chart if you drew it, saved as your hand-in.
figs = {"floor": fig_floor} if "fig_floor" in dir() else {}
ht.record.handin("planet-weighed",
                 ["my_planet", "wobble", "above_floor", "mine_vs_floor", "density", "made_of", "weighed", "in_the_way"],
                 figures=figs)